# **Coding Redundancy**

In [2]:
from collections import Counter
import heapq
import math

# 2x2 grayscale image
image = [
    [100, 100],
    [150, 200]
]

# Step 1: Flatten and count frequencies
pixels = [p for row in image for p in row]
freq = Counter(pixels)
total_symbols = len(pixels)

# Step 2: Fixed-length encoding bits
unique_symbols = len(freq)
fixed_bits_per_symbol = math.ceil(math.log2(unique_symbols))
total_fixed_bits = total_symbols * fixed_bits_per_symbol

# Step 3: Huffman coding
heap = [[wt, [sym, ""]] for sym, wt in freq.items()]
heapq.heapify(heap)
while len(heap) > 1:
    lo = heapq.heappop(heap)
    hi = heapq.heappop(heap)
    for pair in lo[1:]:
        pair[1] = '0' + pair[1]
    for pair in hi[1:]:
        pair[1] = '1' + pair[1]
    heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])
huff_dict = dict(sorted(heap[0][1:]))

# Step 4: Huffman encoding length
total_huffman_bits = sum(freq[sym] * len(code) for sym, code in huff_dict.items())

# Step 5: Compute redundancy and saving
avg_fixed = total_fixed_bits / total_symbols
avg_huffman = total_huffman_bits / total_symbols
redundancy = avg_fixed - avg_huffman
saving = (1 - total_huffman_bits / total_fixed_bits) * 100

# Output
print("Huffman Codes:")
for k, v in huff_dict.items():
    print(f"{k}: {v}")

print(f"\nFixed-length bits per symbol: {fixed_bits_per_symbol}")
print(f"Total Fixed-length bits: {total_fixed_bits}")
print(f"Total Huffman bits: {total_huffman_bits}")
print(f"Redundancy: {redundancy:.2f} bits/symbol")
print(f"Code saving: {saving:.2f}%")

Huffman Codes:
100: 0
150: 10
200: 11

Fixed-length bits per symbol: 2
Total Fixed-length bits: 8
Total Huffman bits: 6
Redundancy: 0.50 bits/symbol
Code saving: 25.00%


# **Spatial Redundancy**

In [3]:
from collections import Counter
import heapq
import math

# Step 1: Original 2x2 grayscale image
image = [
    [100, 100],
    [150, 200]
]

def flatten(img):
    return [p for row in img for p in row]

def huffman_encode(data):
    freq = Counter(data)
    heap = [[wt, [sym, ""]] for sym, wt in freq.items()]
    heapq.heapify(heap)
    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        for pair in lo[1:]:
            pair[1] = '0' + pair[1]
        for pair in hi[1:]:
            pair[1] = '1' + pair[1]
        heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])
    huff_dict = dict(heap[0][1:])
    total_bits = sum(freq[sym] * len(huff_dict[sym]) for sym in freq)
    return total_bits, huff_dict, freq

# Step 2: Huffman encode original image
original_pixels = flatten(image)
original_bits, original_codes, original_freq = huffman_encode(original_pixels)

# Step 3: Create difference image (left neighbor predictor)
def get_diff_image(img):
    diff = []
    for row in img:
        row_diff = [row[0]]  # first pixel stays the same
        for i in range(1, len(row)):
            row_diff.append(row[i] - row[i-1])
        diff.append(row_diff)
    return diff

diff_image = get_diff_image(image)
diff_pixels = flatten(diff_image)

# Step 4: Huffman encode difference image
diff_bits, diff_codes, diff_freq = huffman_encode(diff_pixels)

# Step 5: Compute spatial redundancy saving
saving = (1 - diff_bits / original_bits) * 100

# Output
print("Original Image Huffman Coding:")
for k, v in original_codes.items():
    print(f"{k}: {v}")
print(f"Total bits: {original_bits}\n")

print("Difference Image Huffman Coding:")
for k, v in diff_codes.items():
    print(f"{k}: {v}")
print(f"Total bits: {diff_bits}\n")

print(f"Spatial Redundancy Saving: {saving:.2f}%")

Original Image Huffman Coding:
100: 0
150: 10
200: 11
Total bits: 6

Difference Image Huffman Coding:
0: 00
50: 01
100: 10
150: 11
Total bits: 8

Spatial Redundancy Saving: -33.33%


# **Irrelavent Redundancy**

In [4]:
import numpy as np
from scipy.fftpack import dct, idct
from collections import Counter
import heapq

# 1. Original 2x2 grayscale image
image = np.array([
    [100, 100],
    [150, 200]
], dtype=np.float32)

# Flattened original image
original_pixels = image.flatten().astype(int).tolist()

# 2. Huffman on original (no psychovisual reduction)
def huffman_encode(data):
    freq = Counter(data)
    heap = [[wt, [sym, ""]] for sym, wt in freq.items()]
    heapq.heapify(heap)
    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        for pair in lo[1:]: pair[1] = '0' + pair[1]
        for pair in hi[1:]: pair[1] = '1' + pair[1]
        heapq.heappush(heap, [lo[0]+hi[0]] + lo[1:] + hi[1:])
    huff_dict = dict(heap[0][1:])
    total_bits = sum(freq[sym] * len(huff_dict[sym]) for sym in freq)
    return total_bits, huff_dict, freq

original_bits, original_codes, _ = huffman_encode(original_pixels)

# 3. Apply DCT (2D)
def dct2(a):
    return dct(dct(a.T, norm='ortho').T, norm='ortho')

dct_image = dct2(image)

# 4. Quantize DCT (simulate psychovisual loss)
Q = 10  # Quantization step
quant_dct = np.round(dct_image / Q).astype(int)
quant_dct_flat = quant_dct.flatten().tolist()

# 5. Huffman encode quantized DCT
quant_bits, quant_codes, _ = huffman_encode(quant_dct_flat)

# 6. Calculate savings
saving = (1 - quant_bits / original_bits) * 100

# Print Results
print("Original Image Huffman Codes:")
for k, v in original_codes.items():
    print(f"{k}: {v}")
print(f"Total bits (original): {original_bits}\n")

print("Quantized DCT Huffman Codes:")
for k, v in quant_codes.items():
    print(f"{k}: {v}")
print(f"Total bits (quantized DCT): {quant_bits}\n")

print(f"Irrelevant Redundancy Saving: {saving:.2f}%")

Original Image Huffman Codes:
100: 0
150: 10
200: 11
Total bits (original): 6

Quantized DCT Huffman Codes:
-8: 00
-3: 01
2: 10
28: 11
Total bits (quantized DCT): 8

Irrelevant Redundancy Saving: -33.33%


# **Calculation of Entropy of an image**

In [5]:
import numpy as np
from collections import Counter
import math

def calculate_entropy(image):
    # Flatten the image to 1D
    pixels = image.flatten().tolist()

    # Count frequency of each pixel value
    freq = Counter(pixels)
    total = len(pixels)

    # Calculate entropy
    entropy = -sum((count/total) * math.log2(count/total) for count in freq.values())
    return entropy

# Example: 2x2 grayscale image
image = np.array([
    [100, 100],
    [150, 200]
], dtype=np.uint8)

entropy = calculate_entropy(image)
print(f"Entropy of the image: {entropy:.4f} bits per pixel")

Entropy of the image: 1.5000 bits per pixel


# **Calculation of fidelity of an image**

In [6]:
import numpy as np
import math

def calculate_mse_psnr(original, compressed):
    # Ensure input arrays are float type
    original = original.astype(np.float32)
    compressed = compressed.astype(np.float32)

    mse = np.mean((original - compressed) ** 2)

    if mse == 0:
        psnr = float('inf')  # No error means infinite PSNR
    else:
        max_pixel = 255.0
        psnr = 10 * math.log10((max_pixel ** 2) / mse)

    return mse, psnr

# Example: original and compressed 2x2 images
original_image = np.array([
    [100, 100],
    [150, 200]
], dtype=np.uint8)

compressed_image = np.array([
    [102, 98],
    [148, 205]
], dtype=np.uint8)

mse, psnr = calculate_mse_psnr(original_image, compressed_image)

print(f"MSE: {mse:.4f}")
print(f"PSNR: {psnr:.2f} dB")

MSE: 9.2500
PSNR: 38.47 dB


# **Unary Encoding**

In [7]:
def unary_encode(n):
    return '1' * n + '0'

# Test
for i in range(5):
    print(f"{i}: {unary_encode(i)}")

0: 0
1: 10
2: 110
3: 1110
4: 11110


# **Golomb Encoding**

In [8]:
import math

def unary_encode(n):
    return '1' * n + '0'

def truncated_binary_encode(r, m):
    b = math.ceil(math.log2(m))
    threshold = (1 << b) - m
    if r < threshold:
        return format(r, f'0{b - 1}b')
    else:
        r += threshold
        return format(r, f'0{b}b')

def golomb_encode(n, m):
    q = n // m
    r = n % m
    return unary_encode(q) + truncated_binary_encode(r, m)

# Test
m = 3
for n in range(10):
    print(f"{n}: {golomb_encode(n, m)}")

0: 00
1: 010
2: 011
3: 100
4: 1010
5: 1011
6: 1100
7: 11010
8: 11011
9: 11100
